# Liam MELTS Data Pipeline Prototype

Prototype scope: wrangled -> Liam input -> MELTS thermodynamic workbook -> extracted wide/long tables.
No pressure fitting analysis is run in this notebook unless explicitly enabled.

In [ ]:
from __future__ import annotations

from datetime import datetime
from pathlib import Path
import importlib
import os
import re
import sys

import pandas as pd
from IPython.display import display

from sci_helpers import (
    DEFAULT_REQUIRED_DATASET_OXIDES,
    DEFAULT_REQUIRED_OUTPUT_OXIDES,
    extract_calls_long,
    extract_calls_wide,
    extract_workbook_tables,
    write_liam_input_csv,
    write_workbook_tables,
)

# 1) Config
REPO_ROOT = Path.cwd().resolve()
WRANGLED_CSV = REPO_ROOT / "sci-data/wrangled-outputs/wrangled_KCP-109-C_compositions.csv"
SAMPLE_ID = "KCP-109-B"

LIAM_CODE_DIR = REPO_ROOT / "vendor/LeiTesting"
LIAM_MODULE_NAME = "MeltsHelperFunctions"  # or "MeltsFP"

RUN_PRESSURE_ANALYSIS = False
MAX_COMPOSITION_WORKERS = 1
MAX_PRESSURE_WORKERS = 1

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = REPO_ROOT / "outputs/melts-data-only-runs" / RUN_ID
LIAM_INPUT_CSV = RUN_DIR / "liam-input" / f"liam_input_{WRANGLED_CSV.stem}__{SAMPLE_ID}.csv"
TABLES_DIR = RUN_DIR / "tables"

MELTS_PARAMS = {
    "Model": "rhyolite-MELTS_v1.0.x",
    "Calculation": "QF_P_Calc",
    "T1": 1100,
    "T2": 700,
    "ΔT": 1,
    "T unit": "C",
    "P1": 400,
    "P2": 50,
    "ΔP": 25,
    "P unit": "MPa",
    "fO2 offset": 0,
    "fO2 buffer": "NNO",
    "fO2 constraint": True,
    "ΔH": 0.5,
    "ΔV": 0,
    "ΔS": 0,
}

FIXED_OXIDE_OVERRIDES = {
    "Fe2O3": 0.0,
    "Cr2O3": 0.0,
    "NiO": 0.0,
    "CoO": 0.0,
    "H2O": 13.0,
    "CO2": 0.0,
    "SO3": 0.0,
    "Cl2O-1": 0.0,
    "F2O -1": 0.0,
}

REQUIRED_DATASET_OXIDES = list(DEFAULT_REQUIRED_DATASET_OXIDES)
REQUIRED_OUTPUT_OXIDES = list(DEFAULT_REQUIRED_OUTPUT_OXIDES)

RUN_DIR.mkdir(parents=True, exist_ok=True)
(RUN_DIR / "liam-input").mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

print("Repo root:", REPO_ROOT)
print("Run dir:", RUN_DIR)
print("Wrangled CSV:", WRANGLED_CSV)
print("Sample:", SAMPLE_ID)
print("Liam module:", LIAM_MODULE_NAME)
print("run_pressure_analysis:", RUN_PRESSURE_ANALYSIS)

In [ ]:
# 2) Convert wrangled -> Liam CSV (single sample)
liam_input_csv = write_liam_input_csv(
    wrangled_csv=WRANGLED_CSV,
    sample_id=SAMPLE_ID,
    output_csv=LIAM_INPUT_CSV,
    melts_params=MELTS_PARAMS,
    fixed_oxide_overrides=FIXED_OXIDE_OVERRIDES,
    required_dataset_oxides=REQUIRED_DATASET_OXIDES,
    required_output_oxides=REQUIRED_OUTPUT_OXIDES,
)
print("Liam input CSV:", liam_input_csv)
display(pd.read_csv(liam_input_csv).head(40))

In [ ]:
# 3) Run MELTS thermodynamic loop in data-only mode
if str(LIAM_CODE_DIR) not in sys.path:
    sys.path.insert(0, str(LIAM_CODE_DIR))

liam_module = importlib.import_module(LIAM_MODULE_NAME)
liam_module = importlib.reload(liam_module)

cwd_before = Path.cwd()
os.chdir(RUN_DIR)
try:
    results = liam_module.parallel_melts_main_loop(
        compositions_csv=str(liam_input_csv),
        max_composition_workers=MAX_COMPOSITION_WORKERS,
        max_pressure_workers=MAX_PRESSURE_WORKERS,
        verbose=True,
        run_pressure_analysis=RUN_PRESSURE_ANALYSIS,
    )
finally:
    os.chdir(cwd_before)

results_df = pd.DataFrame(results)
display(results_df)

workbook_paths = []
for result in results:
    filename = result.get("filename")
    if not filename:
        continue
    workbook_path = Path(filename)
    if not workbook_path.is_absolute():
        workbook_path = RUN_DIR / workbook_path
    if workbook_path.exists():
        workbook_paths.append(workbook_path.resolve())

print("Workbook paths:")
for wp in workbook_paths:
    print(" -", wp)

In [ ]:
# 4) Inspect workbook sheets and preview core tables
if not workbook_paths:
    raise RuntimeError("No workbook outputs found. Check vendor logs/results table above.")

workbook_path = workbook_paths[0]
tables = extract_workbook_tables(workbook_path)
print("Sheets:", list(tables.keys()))

for sheet_name in ["init_cond", "system", "liquid"]:
    if sheet_name in tables:
        print(f"\nPreview: {sheet_name}")
        display(tables[sheet_name].head(10))

In [ ]:
# 5) Export workbook tables + wide/long call tables
table_csvs = write_workbook_tables(workbook_path=workbook_path, output_dir=TABLES_DIR)
calls_wide = extract_calls_wide(workbook_path)
calls_long = extract_calls_long(workbook_path)

calls_wide_path = RUN_DIR / "calls_wide.csv"
calls_long_path = RUN_DIR / "calls_long.csv"
calls_wide.to_csv(calls_wide_path, index=False)
calls_long.to_csv(calls_long_path, index=False)

print("Table CSV exports:")
for p in table_csvs:
    print(" -", p)
print("calls_wide:", calls_wide_path, "rows=", len(calls_wide))
print("calls_long:", calls_long_path, "rows=", len(calls_long))

display(calls_wide.head(20))
display(calls_long.head(20))

In [ ]:
# 6) Pressure prep study: inspect pressure_calc dependencies from vendor code
vendor_file = LIAM_CODE_DIR / f"{LIAM_MODULE_NAME}.py"
vendor_code = vendor_file.read_text(encoding="utf-8")

start = vendor_code.find("def pressure_calc")
if start < 0:
    raise RuntimeError(f"pressure_calc not found in {vendor_file}")
end = vendor_code.find("\ndef ", start + 1)
if end < 0:
    end = len(vendor_code)
pressure_code = vendor_code[start:end]

required_sheet_literals = sorted(set(re.findall(r"sheet_name='([^']+)'", pressure_code)))
required_logic_tokens = {
    "requires_quartz_sheet": "quartz" in pressure_code.lower(),
    "requires_feldspar_prefix": "startswith('feldspar')" in pressure_code,
    "requires_init_cond": "init_cond" in pressure_code,
    "reads_pressure_from_init_cond": "iloc[4, 4]" in pressure_code,
    "reads_temperature_pressure_columns": all(
        token in pressure_code for token in ["'T (C)'", "'P (MPa)'"]
    ),
}

print("Detected pressure_calc sheet literals:", required_sheet_literals)
print("Detected pressure_calc requirements:")
for k, v in required_logic_tokens.items():
    print(f" - {k}: {v}")

print("\npressure_calc code preview (first 140 lines):")
for i, line in enumerate(pressure_code.splitlines()[:140], start=1):
    print(f"{i:03d}: {line}")